## SQL AI Agent in Action

General Requirements:
- OpenAI API Python SDK compatibility 
- Arguments: 
   - API arguments
   - Table name
- Input: question
- Output: table

<br>
<figure>
 <img src="assets/sql_agent.jpg" width="100%" align="center"/></a>
<figcaption> SQL AI Agent Architecture </figcaption>
</figure>

<br>

In [12]:
import sys
import os
import pandas as pd
import duckdb as db

In [13]:
sys.path.append("../")
from sql_ai_agent.api_handler import SqlAgent

In [14]:
nbi_2025_ak = pd.read_csv("../data/nbi/AK25.csv")

db.register("nbi_2025_ak", nbi_2025_ak)

In [15]:
openai_base_url = "https://api.openai.com/v1/"
openai_api_key = os.environ.get("OPENAI_API_KEY")

openai_sql_agent = SqlAgent(
    api_key=openai_api_key,
    base_url=openai_base_url,
    model="gpt-5",
    tbl_name="nbi_2025_ak",
    max_token=5000,
)

In [16]:
question = "What are the total number of bridges by county? Return the top 10 results ordered by the number of bridges."

In [17]:
openai_sql_agent.ask_question(question=question)

SELECT
  COUNTY_CODE_003 AS county_code,
  COUNT(*) AS bridge_count
FROM nbi_2025_ak
WHERE COUNTY_CODE_003 IS NOT NULL
GROUP BY COUNTY_CODE_003
ORDER BY bridge_count DESC
LIMIT 10;
┌─────────────┬──────────────┐
│ county_code │ bridge_count │
│   double    │    int64     │
├─────────────┼──────────────┤
│       198.0 │          226 │
│       195.0 │          137 │
│        20.0 │          134 │
│       170.0 │          121 │
│        90.0 │          112 │
│       130.0 │          107 │
│       105.0 │          103 │
│       122.0 │          102 │
│       290.0 │           94 │
│        63.0 │           65 │
├─────────────┴──────────────┤
│ 10 rows          2 columns │
└────────────────────────────┘



Review the system message:

In [19]:
print(openai_sql_agent.system.system)

Given the following SQL table, your job is to write queries given a user’s request. Return just the SQL query as plain text, without additional text, and don't use markdown format.

CREATE TABLE nbi_2025_ak (STATE_CODE_001 BIGINT, STRUCTURE_NUMBER_008 VARCHAR, RECORD_TYPE_005A BIGINT, ROUTE_PREFIX_005B BIGINT, SERVICE_LEVEL_005C BIGINT, ROUTE_NUMBER_005D VARCHAR, DIRECTION_005E BIGINT, HIGHWAY_DISTRICT_002 VARCHAR, COUNTY_CODE_003 DOUBLE, PLACE_CODE_004 BIGINT, FEATURES_DESC_006A VARCHAR, CRITICAL_FACILITY_006B DOUBLE, FACILITY_CARRIED_007 VARCHAR, LOCATION_009 VARCHAR, MIN_VERT_CLR_010 DOUBLE, KILOPOINT_011 DOUBLE, BASE_HWY_NETWORK_012 DOUBLE, LRS_INV_ROUTE_013A DOUBLE, SUBROUTE_NO_013B DOUBLE, LAT_016 BIGINT, LONG_017 BIGINT, DETOUR_KILOS_019 BIGINT, TOLL_020 BIGINT, MAINTENANCE_021 BIGINT, OWNER_022 BIGINT, FUNCTIONAL_CLASS_026 BIGINT, YEAR_BUILT_027 BIGINT, TRAFFIC_LANES_ON_028A BIGINT, TRAFFIC_LANES_UND_028B BIGINT, ADT_029 BIGINT, YEAR_ADT_030 BIGINT, DESIGN_LOAD_031 VARCHAR, APP

Review the user message:

In [21]:
print(openai_sql_agent.user)

Write a SQL query that returns: What are the total number of bridges by county? Return the top 10 results ordered by the number of bridges.
